In [1]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from pandas import read_sql
from sqlite3 import connect
import seaborn as sns
from datetime import datetime

## Dashboard Summary
This summary will be broken into parts by the different charts we have presented

#### Sentiment Distribution per Length
- This plot shows if there is any noticeable difference in the distribution of lengths in the articles/posts depending on their sentiment towards A.I. I.e., are articles that talk positively about A.I. longer or shorter than the negative ones, and vice versa
- Our findings with this plot, even when ignoring outliers, is that there is no significant difference in the lengths of posts or articles

#### Sentiment Distribution per Topic
- This plot shows whether developer posts that focus on certain topics, tend to be more positive or negative compared to other posts/topics
- This chart found that most topics had similar sentiment distributions, other than topic 4, which included discussion of AI in enterprise settings. The sentiment distribution for this topic had almost 10% points more positive posts than compared to the other topics

#### Topic Discussion Over Time
- This plot illustrates the trends in terms of topical conversation around AI. Since a lot of developer posts in our dataset didn't include dates, the trend lines past 2024 seem to be the only ones worth analyzing
- Our plot shows that topics 0 and 4 had a steady rise, and continue to be rising, while topics 2 and 3 had sharp jumps in the last few months, which potentially could be outliers in the long run, bit is difficult to know without seeing what data for future months is like

#### Sentiment Changes Over Time
- This chart aims to show how the sentiment on A.I. has changed over the early 2020s, considering both our DevPosts dataset, but also our News Articles dataset.
- Our findings show that positive sentiment has declined over time, while negative has increased. Although negative sentiment is on the rise, it has yet to take over the positive posts/articles, but at this rate it seems likely to happen within a year or so

### Sentiment Distribution per Length

In [2]:
# Bar chart showing distribution of article length by sentiment
def chart_sentiment_dist_over_length(table_name, column, date_column, sentiment_threshold=0, length_threshold=500000):
    query = f"""SELECT {column}, IF(roberta_pos_score - roberta_neg_score > {sentiment_threshold}, 'positive', 'negative') sentiment, strftime('%Y-%m', {date_column}) month_released
                FROM {table_name}
                WHERE month_released BETWEEN "2021-01" AND "2025-12"
                ORDER BY month_released, strftime('%m', {date_column});"""
    conn = connect("db.sqlite")

    df = read_sql(query, conn)
    df["length"] = df[column].apply(len)
    df = df[df["length"] < length_threshold]
    return df

In [3]:
@widgets.interact(sentiment_threshold=(-1, 1, 0.1), news_length_threshold=(10000, 100000, 5000), devposts_length_threshold=(10000, 100000, 5000))
def display_sen_by_len(sentiment_threshold, news_length_threshold, devposts_length_threshold):
    articles_df = chart_sentiment_dist_over_length("modified_articles", "text", date_column="date", sentiment_threshold=sentiment_threshold, length_threshold=news_length_threshold)
    dp_df = chart_sentiment_dist_over_length("DevPosts", "body_text", date_column="published_at", sentiment_threshold=sentiment_threshold, length_threshold=devposts_length_threshold)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.boxplot(x="sentiment", y='length', data=articles_df, ax=axes[0])

    sns.boxplot(x="sentiment", y='length', data=dp_df, ax=axes[1])
    for ax in axes:
        ax.set_xlabel("")
        ax.set_ylabel("")
    axes[0].set_title("News Articles Sentiment Distribution by Length")
    axes[1].set_title("DevPosts Sentiment Distribution by Length")
    fig.supylabel("Length (chars)")
    fig.supxlabel("Sentiment")

interactive(children=(FloatSlider(value=0.0, description='sentiment_threshold', max=1.0, min=-1.0), IntSlider(…

### Sentiment Distribution per Topic

In [ ]:
topic_list = ["code, software, development", "models, model, google", "just, like, work", "learning, machine, artificial", "enterprise, 2025, artificial"]

In [13]:
# Interactive pie chart to show % of articles/posts that are positive or negative with tabs to switch between different topics
def chart_sent_by_topic(topic_choice):
    query = f"""SELECT SUM(IF(roberta_pos_score - roberta_neg_score >= 0, 1, 0)) * 100 / count(*) positive,
            SUM(IF(roberta_pos_score - roberta_neg_score <= 0, 1, 0)) * 100 / count(*) negative
                    FROM DevPosts
                    WHERE dominant_topic {'=' if topic_choice != -1 else '!='} {topic_choice}
                    GROUP BY dominant_topic;"""
    conn = connect("db.sqlite")

    df = read_sql(query, conn).transpose()
    conn.close()
    return df

In [25]:
_df = chart_sent_by_topic(-1)
_df['summed'] = _df.sum(axis=1)
_df = _df[["summed"]]

print(_df)

          summed
positive     470
negative     416


In [27]:
@widgets.interact(topic_choice=["All", "0", "1", "2", "3", "4"])
def display_sen_by_len(topic_choice):
    if topic_choice == "All":
        topic_choice = -1

    df = chart_sent_by_topic(topic_choice)
    if topic_choice == -1:
        df[0] = df.sum(axis=1)
        df = df[[0]]

    plt.pie(df[0], labels=["Positive", "Negative"], autopct="%1.1f%%")
    if topic_choice != -1:
        plt.title("Developer Posts Sentiment Analysis Percentage by Topic")
        plt.figtext(0.33, 0.05, f"Topic Keywords: {topic_list[int(topic_choice)]}")
    else:
        plt.title("Developer Posts Sentiment Analysis Percentage")

    plt.show()

interactive(children=(Dropdown(description='topic_choice', options=('All', '0', '1', '2', '3', '4'), value='Al…

### Topic discussion over time

In [28]:
# Line chart with n lines per n topics popularity over time. Maybe have tabs per topic instead of lines on the same plot
def chart_topic_disc_over_time(post_per_month_threshold):
    query = f"""SELECT strftime('%Y-%m', published_at) month_released,
    SUM(IF(dominant_topic = 0, 1, 0)) topic_0,
    SUM(IF(dominant_topic = 1, 1, 0)) topic_1,
    SUM(IF(dominant_topic = 2, 1, 0)) topic_2,
    SUM(IF(dominant_topic = 3, 1, 0)) topic_3,
    SUM(IF(dominant_topic = 4, 1, 0)) topic_4
                    FROM DevPosts
                    where published_at < "2026-03-01"
                    GROUP BY month_released
                    HAVING COUNT(*) > {post_per_month_threshold}
                    ORDER BY month_released;"""

    conn = connect("db.sqlite")

    df = read_sql(query, conn)
    conn.close()
    df["month_released"] = df["month_released"].apply(lambda x: datetime.strptime(x, "%Y-%m"))
    for i in range(5):
        plt.plot(df["month_released"], df[f"topic_{i}"], label=f"Topic {i}: {topic_list[i]} ")


In [29]:
@widgets.interact(post_per_month_threshold=widgets.IntSlider(min=0, max=100, step=5, value=5))
def display_topic_disc_over_time(post_per_month_threshold):
    chart_topic_disc_over_time(post_per_month_threshold)
    plt.xlabel("Month")
    plt.ylabel("Topic")
    plt.title("Topic Change over Months")
    plt.xticks(rotation=90)
    plt.legend()
    plt.show()


interactive(children=(IntSlider(value=5, description='post_per_month_threshold', step=5), Output()), _dom_clas…

### Sentiment changes over time

In [9]:
# Line chart with 2 lines, positive and negative sentiment over time
def chart_sentiment_change_over_months(table_name, column, threshold=0, post_per_month_threshold=5):
    query = f"""SELECT strftime('%Y-%m', {column}) month_released,
    SUM(IF(roberta_pos_score - roberta_neg_score >= {threshold}, 1, 0)) * 100 / count(*) positive,
    SUM(IF(roberta_pos_score - roberta_neg_score <= -{threshold}, 1, 0)) * 100 / count(*) negative
                FROM {table_name}
                WHERE month_released BETWEEN "2021-01" AND "2025-12"
                GROUP BY month_released
                HAVING COUNT(*) > {post_per_month_threshold}
                ORDER BY month_released;"""
    conn = connect("db.sqlite")

    df = read_sql(query, conn)
    df["month_released"] = df["month_released"].apply(lambda x: datetime.strptime(x, "%Y-%m"))
    return df

In [10]:
@widgets.interact(sentiment_threshold=(-1, 1, 0.1), post_per_month_threshold=widgets.IntSlider(min=0, max=100, step=5, value=5))
def display_sent_change_over_time(sentiment_threshold, post_per_month_threshold):
    articles_df = chart_sentiment_change_over_months("modified_articles", "date", threshold=sentiment_threshold, post_per_month_threshold=post_per_month_threshold)
    dp_df = chart_sentiment_change_over_months("DevPosts", "published_at", threshold=sentiment_threshold, post_per_month_threshold=post_per_month_threshold)

    plt.plot(articles_df["month_released"], articles_df["negative"], label="Negative News Articles")
    plt.plot(articles_df["month_released"], articles_df["positive"], label="Positive News Articles")

    plt.plot(dp_df["month_released"], dp_df["negative"], label="Negative DevPosts")
    plt.plot(dp_df["month_released"], dp_df["positive"], label="Positive DevPosts")
    plt.xlabel("Year")
    plt.ylabel("Sentiment %")
    plt.title("Sentiment Change over Year")
    plt.legend()
    plt.show()

interactive(children=(FloatSlider(value=0.0, description='sentiment_threshold', max=1.0, min=-1.0), IntSlider(…